# 01. Откуда берутся логи

Лог — журнал событий программы. Веб-сервер записывает получение запроса, обработчик файлов — начало чтения и ошибки, фоновая задача — завершение работы.

Запись появляется, когда код программы вызывает функцию логирования. В Python это, например, `logger.info("Начало чтения")`. Настройки определяют, какие сообщения попадут в файл, консоль или сервис сбора логов.

Типичная запись содержит время, уровень важности, источник и сообщение:

```text
2026-10-07T12:00:00+03:00 INFO loader Начало чтения файла
```

В ЛР № 2 пакет `streamstats` анализирует готовые события, сохранённые в CSV или JSONL, и создаёт статистический отчёт. Такие файлы могут быть выгрузкой журнала или учебным набором данных.

## 02. Этапы обработки

**Чтение → разбор → валидация → статистика → отчёт.**

- Чтение получает текст из файла.
- Разбор превращает запись в значения Python.
- Валидация проверяет наличие полей, их типы и допустимые значения.
- Расчёт статистики накапливает итоги по корректным событиям.
- Отчёт сохраняет эти итоги в JSON.

Синтаксически правильный JSON может не соответствовать модели события: например, в нём может отсутствовать `source`.

## 03. Файл, путь и `with`

`Path("data/participants.txt")` хранит путь; `.open()` возвращает открытый файл. Относительный путь отсчитывается от рабочего каталога программы.

**`with`** работает с контекстным менеджером — объектом с действиями входа и выхода из блока. У открытого файла выход закрывает файл, в том числе при исключении. `as file` связывает открытый файл с именем `file`. Само исключение при этом не исчезает.

При обходе файла строки поступают по очереди. `read()` без ограничения и `readlines()` собирают всё содержимое в память.

`encoding="utf-8"` задаёт кодировку — правило преобразования байтов в символы. Режим `"r"` читает файл, `"w"` записывает с очисткой прежнего содержимого, `"a"` добавляет в конец. В примере используется режим чтения по умолчанию.

In [1]:
from pathlib import Path

with Path("data/participants.txt").open(encoding="utf-8") as file:
    for number, line in enumerate(file, start=1):
        print(number, line.rstrip("\n"))

1 Анна
2 Илья
3 Ольга


## 05. Итератор и генератор

**Итерируемый объект** предоставляет элементы для обхода. **Итератор** выдаёт следующий элемент по запросу `next()` и помнит положение в обходе.

**Генератор** — особый итератор. В примере его создаёт функция с `yield`. `yield` выдаёт значение и приостанавливает функцию; следующий запрос продолжает её выполнение. Тело функции начинает работать при первом запросе, а не при создании генератора. После исчерпания тот же генератор не начинает обход заново.

In [2]:
def badges(names):
    for name in names:
        print("Готовим:", name)
        yield name.upper()


items = badges(("Анна", "Илья"))
print("Генератор создан")
print(next(items))
print(list(items))
print(list(items))

Генератор создан
Готовим: Анна
АННА
Готовим: Илья
['ИЛЬЯ']
[]


## 07. CSV

**CSV** хранит поля в столбцах; заголовок задаёт их имена. Модуль `csv` учитывает кавычки и разделители. Запятая внутри `"Книги, чтение"` относится к названию и не разделяет поля.

`csv.DictReader` выдаёт словарь для каждой записи; ключи соответствуют заголовкам столбцов. Значения CSV по умолчанию являются строками.

`StringIO` представляет короткую строку как текстовый поток для демонстрации. `newline=""` оставляет обработку переводов строк CSV-reader. CSV-запись может занимать несколько физических строк.

In [3]:
import csv
from io import StringIO

csv_text = 'name,club\nАнна,"Книги, чтение"\n'
for row in csv.DictReader(StringIO(csv_text, newline="")):
    print(row["name"], "—", row["club"])

Анна — Книги, чтение


## 09. JSON и JSONL

**JSON** — текстовый формат данных. Объект JSON превращается в словарь Python, JSON-строка — в `str`, целое число — в `int`.

**JSONL** хранит по одному JSON-значению в каждой строке. В лабораторной ожидаются объекты событий. `json.loads(line)` разбирает текст одной записи.

В примере `3` становится числом, а `"три"` остаётся строкой. `type(value).__name__` показывает название типа. Обе записи имеют корректный синтаксис JSON, но это ещё не проверка допустимости их полей.

In [4]:
import json

jsonl_text = '{"name":"Анна","visits":3}\n{"name":"Илья","visits":"три"}\n'
for line in StringIO(jsonl_text):
    record = json.loads(line)
    visits = record["visits"]
    print(record["name"], visits, type(visits).__name__)

Анна 3 int
Илья три str


## 11. Валидация события

Валидация проверяет состав, типы и значения данных. По условию события нужны:

- `timestamp` — время ISO 8601;
- `level` — `DEBUG`, `INFO`, `WARNING`, `ERROR` или `CRITICAL`;
- `source` — непустая строка;
- `message` — строка.

Отсутствующее поле и пустое значение — разные случаи. 

## 12. Время и часовой пояс

**Временная метка** записывает дату и время события. В `2026-10-07T12:00:00+03:00` после даты и буквы `T` идёт время; `+03:00` означает смещение на три часа вперёд относительно UTC.

**UTC** — единая точка отсчёта времени. `12:00+03:00` и `09:00+00:00` описывают один момент. Текст этих записей различается, поэтому для сравнения времени его сначала разбирают в объекты `datetime`.

In [5]:
from datetime import datetime

local_text = "2026-10-07T12:00:00+03:00"
utc_text = "2026-10-07T09:00:00+00:00"
print(local_text == utc_text)
print(datetime.fromisoformat(local_text) == datetime.fromisoformat(utc_text))

False
True


## 14. Агрегация и память

**Агрегация** — накопление общих итогов. Словарь `{"шахматы": 2, "робототехника": 1}` хранит числа участников по кружкам вместо всех анкет.

Память такого словаря растёт с числом разных кружков. В лабораторной аналогично растут счётчики разных источников.

Потоковая обработка сохраняет текущую запись, буферы и итоги. Чтение по строкам с последующим накоплением всех событий в списке не выполняет требование потоковости. При переходе к следующему файлу общая статистика должна сохраняться.

## 15. Порядок при равенстве

**Детерминированный результат** воспроизводится на одинаковых входных данных. Для top-5 критерии требуют определённого порядка при равных счётчиках.

Дополнительным правилом может быть порядок названий или другой описанный способ. Конкретный вторичный ключ условием не задан. Стабильная сортировка сохраняет прежний порядок равных элементов; этот порядок может зависеть от входа.

## 16. Обработка ошибочной строки

По умолчанию программа останавливается, сообщает файл и строку ошибки и завершает CLI с кодом `2`.

С `--skip-invalid` неверная строка пропускается, в лог записывается `WARNING`, число пропусков попадает в отчёт. Корректные записи после ошибки продолжают учитываться. Номер ошибочной JSONL-строки относится к исходному файлу, а не к числу успешно прочитанных событий.

## 17. Семейства исключений

Базовый тип объединяет связанные ошибки. `EmptyNameError` наследуется от `RegistrationError`, поэтому `except RegistrationError` обрабатывает и эту частную ошибку. Обработчик сохраняет конкретный тип и сообщение исключения.

Для ЛР2 нужны общий `StreamStatsError` и отдельные типы ошибок формата, события, времени и конфигурации CLI.

In [6]:
class RegistrationError(Exception):
    pass


class EmptyNameError(RegistrationError):
    pass


try:
    raise EmptyNameError("Имя участника не заполнено")
except RegistrationError as error:
    print(type(error).__name__, "—", error)

EmptyNameError — Имя участника не заполнено


## 19. Логгер, обработчик и отчёт

**Логгер** принимает сообщения от кода программы. **Обработчик** направляет записи в файл или поток вывода. **Форматтер** задаёт вид записи: время, имя, уровень и текст. Настройка уровня отбирает сообщения нужной важности.

`logger.warning()` записывает предупреждение, `logger.error()` — сообщение об ошибке. Эти вызовы сами не возбуждают исключение и не останавливают программу.

Лог описывает ход обработки; отчёт хранит итоговые данные. **Сериализация** переводит значения Python в формат хранения. `json.dumps()` создаёт JSON-текст; `ensure_ascii=False` сохраняет кириллицу читаемыми символами.

In [7]:
import logging
import sys

logger = logging.getLogger("lab2.demo")
logger.setLevel(logging.INFO)
logger.propagate = False
for old_handler in logger.handlers[:]:
    logger.removeHandler(old_handler)
    old_handler.close()

handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(levelname)s | %(name)s | %(message)s"))
logger.addHandler(handler)

logger.debug("Подробности чтения")
logger.info("Начинаем обработку файла")
logger.warning("В строке 3 пропущено имя")
logger.error("Не удалось открыть следующий файл")
print("Выполнение продолжается")

INFO | lab2.demo | Начинаем обработку файла


WARNING | lab2.demo | В строке 3 пропущено имя


ERROR | lab2.demo | Не удалось открыть следующий файл


Выполнение продолжается


In [8]:
summary = {"кружок": "Робототехника", "участников": 2}
report_text = json.dumps(summary, ensure_ascii=False)
print(report_text)
print(json.loads(report_text) == summary)

{"кружок": "Робототехника", "участников": 2}
True



# Лабораторная работа 2. Потоковый анализатор файлов

## Результат

Пакет `streamstats`, читающий события из CSV и JSONL и создающий статистический отчёт.

## Модель события

Обязательные поля:

```text
timestamp: ISO 8601
level: DEBUG | INFO | WARNING | ERROR | CRITICAL
source: непустая строка
message: строка
```

## CLI

```text
python -m streamstats analyze INPUT [INPUT ...] --format csv|jsonl --output REPORT.json
python -m streamstats analyze INPUT --format jsonl --skip-invalid
```

## Обязательная функциональность

- Читать входные файлы построчно.
- Не использовать `read()` без ограничения и `readlines()` для всего файла.
- Поддержать CSV с заголовком и JSONL.
- Валидировать обязательные поля.
- Посчитать общее число событий.
- Посчитать количество событий каждого уровня.
- Посчитать количество событий каждого источника.
- Вывести top-5 источников ошибок уровня ERROR/CRITICAL.
- Найти первую и последнюю временную метку.
- Сформировать JSON-отчёт.
- Объединять статистику нескольких файлов.

## Ошибочные строки

Режим по умолчанию: остановить обработку, вывести номер файла и строки, завершить CLI с кодом `2`.

Режим `--skip-invalid`: пропустить строку, записать WARNING в лог, добавить число пропущенных строк в отчёт.

## Исключения

Создать базовое `StreamStatsError` и отдельные ошибки:

- неподдерживаемый формат;
- неверное событие;
- неверная временная метка;
- ошибка конфигурации CLI.

Не перехватывать `Exception` без повторного возбуждения или преобразования в понятную ошибку на границе CLI.

## Структура

```text
lab_02/
  pyproject.toml
  README.md
  src/streamstats/
    __init__.py
    __main__.py
    models.py
    parsers.py
    analysis.py
    report.py
    errors.py
  tests/
```

## Тесты

- Unit tests обоих parsers.
- Тест пустого файла.
- Тест повреждённой строки в обоих режимах.
- Тест нескольких файлов.
- Тест Unicode.
- Integration test CLI с временным каталогом.

## Команды проверки

```bash
python -m pytest
ruff check .
python -m streamstats --help
```

## Дополнительные задания

- +3: ленивые фильтры по времени, уровню и источнику.
- +3: пользовательский формат через registry parser-функций.
- +2: отчёт CSV без дублирования расчётов.
- +2: измерение памяти на файле не менее 500 МБ.
- +5 Challenge: iterator, отслеживающий дописываемый файл.
- +5 Challenge: обработка бинарного формата через `memoryview`.